# Agent 3 — Security & Threat Detection

> Scan camera footage for six classes of incident — shoplifting,
> scanner bypass, masked entry, slip-and-fall, restricted-area breach,
> altercation — and emit a severity-tagged incident log.

## What this notebook shows

The pattern is exactly the same as Agent 1 (SOP Compliance) but
parameterized over a list of `(search_query, vlm_prompt, severity)`
scenarios. The agent runs cheap retrieval per scenario, then expensive
VLM verification on the union of candidates.

## Endpoints exercised

| Step | Endpoint | What it does |
|---|---|---|
| Retrieve (×N scenarios) | `POST /search` (BY_CLIP, filtering_level=low) | Cast a wide net |
| Reason | `POST /vu/chat/completions` | Strict-JSON verification per candidate |


## Setup

You need:

1. A Memories.ai API key (`sk-mavi-...`) — get one at the [Memories.ai console](https://api-platform.memories.ai/stripe).
2. Python 3.10+ and the `requests` library (`pip install requests`).

Set the key as an environment variable before launching Jupyter, or paste it
inline in the cell below. The same key works across Visual Search, Visual
Intelligence, and Visual Agents — no separate auth per product.


In [ ]:
import os, json, time, requests

# ────────────────────────────────────────────────────────────────────────────
# Auth: the Memories.ai key is a single token used across every product.
# Pass it as the literal `Authorization` header value — no `Bearer` prefix.
# ────────────────────────────────────────────────────────────────────────────
API_KEY = os.environ.get("MEMORIES_API_KEY") or "sk-mavi-..."  # ← paste here if not using env
HEADERS = {"Authorization": API_KEY}

# Hosts: Visual Search and Visual Intelligence live on different domains.
VS_HOST  = "https://api.memories.ai/serve/api/v1"            # Visual Search
VLM_HOST = "https://mavi-backend.memories.ai/serve/api/v2"   # Visual Intelligence (VLM + Visual Agents)

# Default VLM model used for verification / reasoning. Other options include
# `qwen:qwen2.5-vl-72b-instruct`, `nova:amazon.nova-lite-v1:0`, etc. — see
# Memories.ai docs for the full list and per-token pricing.
VLM_MODEL = "gemini:gemini-2.5-flash"


In [ ]:
def search(query, *, video_nos=None, unique_id="default", top_k=10,
           filtering_level="medium", search_type="BY_CLIP",
           datetime_taken=None, tag=None, camera_tag=None,
           latitude=None, longitude=None, max_retries=3):
    """Visual Search — POST /search.

    Returns the list of {videoNo, startTime, endTime, score, ...} matches.

    Filter parameters (all optional, combinable):
      • video_nos       restrict to specific videos (up to 100 ids)
      • datetime_taken  filter to videos captured at-or-after this timestamp
      • tag             filter to videos carrying this user-defined tag
      • camera_tag      filter to videos shot on this camera_model
      • latitude/longitude  GPS proximity filter (must be paired)

    Notes on the envelope: Visual Search responses are wrapped in
    {code, msg, data, success, failed}. code="0000" means OK; the actual
    hit list is in `data`. code="0001" ("network abnormal") is transient
    and we retry it.
    """
    body = {
        "search_param": query,
        "search_type": search_type,                 # "BY_CLIP" or "BY_AUDIO"
        "unique_id": unique_id,                      # namespace within your account
        "top_k": top_k,
        "filtering_level": filtering_level,          # "low"|"medium"|"high"
    }
    if video_nos:        body["video_nos"]     = list(video_nos)
    if datetime_taken:   body["datetime_taken"] = datetime_taken
    if tag:              body["tag"]            = tag
    if camera_tag:       body["camera_tag"]     = camera_tag
    if latitude is not None and longitude is not None:
        body["latitude"]  = latitude
        body["longitude"] = longitude

    for attempt in range(max_retries):
        r = requests.post(f"{VS_HOST}/search", headers=HEADERS, json=body, timeout=60)
        r.raise_for_status()
        envelope = r.json()
        code = envelope.get("code")
        if code == "0000":
            return envelope.get("data") or []
        if code == "0001" and attempt < max_retries - 1:
            # Transient. Backoff and retry.
            time.sleep(0.4 * (2 ** attempt))
            continue
        raise RuntimeError(f"/search failed: code={code} msg={envelope.get('msg')!r}")
    return []


In [ ]:
def vlm_complete(prompt, *, video_url=None, image_url=None, system=None,
                 model=VLM_MODEL, response_json=True, temperature=0.2,
                 max_tokens=1024):
    """Visual Intelligence — POST /vu/chat/completions.

    Calls a Video Language Model (Gemini by default) with text + an
    optional media reference. Returns the assistant's text reply, with
    markdown ```json fences stripped if `response_json=True`.

    Notes on the wire format:
      • `content` MUST be an array even for text-only prompts. A bare
        string is rejected with "Model input cannot be empty".
      • Gemini's response lives in choices[0]["text"]. Other providers
        (Qwen, Nova) use choices[0]["message"]["content"]. We accept both.
      • The endpoint can return HTTP 200 with status="errored" — surface
        that as a typed exception rather than letting JSON parsing fail.
    """
    content = [{"type": "text", "text": prompt}]
    if video_url:
        content.append({"type": "input_file", "file_uri": video_url, "mime_type": "video/mp4"})
    if image_url:
        content.append({"type": "input_file", "file_uri": image_url, "mime_type": "image/jpeg"})

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": content})

    body = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if response_json:
        # Tells Gemini to bias toward JSON output. Other providers ignore this.
        body["extra_body"] = {"metadata": {"response_mime_type": "application/json"}}

    r = requests.post(f"{VLM_HOST}/vu/chat/completions", headers=HEADERS,
                      json=body, timeout=180)
    r.raise_for_status()
    envelope = r.json()
    if envelope.get("status") == "errored" or envelope.get("error"):
        err = envelope.get("error") or {}
        raise RuntimeError(f"VLM error: {err.get('code')} {err.get('message')}")
    choices = envelope.get("choices") or []
    if not choices:
        raise RuntimeError(f"VLM returned no choices: {envelope}")
    # Two shapes observed in the wild.
    text = choices[0].get("text") or (choices[0].get("message") or {}).get("content", "")
    if response_json:
        text = _strip_json_fence(text)
    return text


def _strip_json_fence(text):
    """Gemini often wraps JSON output in ```json ... ``` fences even when
    response_mime_type=application/json. Trim them so json.loads() works."""
    if not text:
        return text
    s = text.strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[1] if "\n" in s else s[3:]
        if s.endswith("```"):
            s = s[:-3].rstrip()
    return s


In [ ]:
def resolve_media_url(video_no):
    """Bridge a videoNo to a public URL the VLM can fetch.

    `/download` streams the raw bytes back to you — it does NOT return a
    hosted URL. To feed a video to /vu/chat/completions you must host it
    yourself. This helper expects either:

      • MEMORIES_MEDIA_URL_TEMPLATE env var (e.g. https://your-cdn/{video_no}.mp4)
      • a MEDIA_URL_MAP dict you populate inline

    If neither is configured, raises so the notebook stops cleanly.
    """
    if video_no in MEDIA_URL_MAP:
        return MEDIA_URL_MAP[video_no]
    tpl = os.environ.get("MEMORIES_MEDIA_URL_TEMPLATE")
    if tpl:
        return tpl.format(video_no=video_no)
    raise RuntimeError(
        f"No media-URL bridge configured for {video_no}. "
        "Set MEMORIES_MEDIA_URL_TEMPLATE or add an entry to MEDIA_URL_MAP."
    )

# Per-notebook overrides: populate this for testing without a CDN.
# A public Memories.ai test asset is included as an example.
MEDIA_URL_MAP = {
    # "VI676024023022092288": "https://storage.googleapis.com/memories-test-data/test_1min.mp4",
}


## Step 1 — declare the security scenarios

Each scenario is `(name, search_query, vlm_prompt, severity)`. Add
scenarios specific to your industry — cannabis cultivation has different
threats than a retail floor.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Scenario:
    name: str
    search_query: str
    vlm_prompt: str
    severity: str  # "low" | "medium" | "high" | "critical"

SCENARIOS = [
    Scenario(
        name="shoplifting_concealment",
        search_query="a person picking up an item and placing it in their pocket or bag without paying",
        vlm_prompt=(
            "Is a person concealing merchandise in clothing or a bag without paying? "
            'Reply JSON: {"violation": bool, "confidence": "low"|"medium"|"high", "notes": str}.'
        ),
        severity="high",
    ),
    Scenario(
        name="masked_entry",
        search_query="a person entering with their face covered by a mask, hood, or scarf",
        vlm_prompt=(
            "Is someone entering with their face deliberately concealed during regular hours? "
            'Reply JSON: {"violation": bool, "confidence": str, "notes": str}.'
        ),
        severity="medium",
    ),
    Scenario(
        name="slip_and_fall",
        search_query="a person slipping or falling to the floor",
        vlm_prompt=(
            "Did someone slip, trip, or fall? Was there a wet-floor sign visible? "
            'Reply JSON: {"violation": bool, "wet_floor_sign_visible": bool, "confidence": str, "notes": str}.'
        ),
        severity="critical",
    ),
    # Add: scanner_bypass, restricted_area_breach, altercation — see the
    # Visual Agents PRD for the full canonical list.
]


## Step 2 — retrieve candidates per scenario

Note `filtering_level="low"` — we want broad recall here because the VLM
will filter false positives. Tuning this for a real deployment is the
biggest cost knob: `low` = more VLM calls, better recall; `high` =
fewer VLM calls, may miss edge cases.


In [ ]:
seed_hits = search("a person", top_k=1, filtering_level=None)
VIDEO_NO  = seed_hits[0]["videoNo"]

candidates = []
for s in SCENARIOS:
    hits = search(s.search_query, video_nos=[VIDEO_NO], top_k=5,
                  filtering_level="low")
    for h in hits:
        candidates.append({"scenario": s, "hit": h})
    print(f"  {s.name:<28} -> {len(hits)} candidates")
print(f"\nTotal candidates across all scenarios: {len(candidates)}")


## Step 3 — VLM verification per candidate

Each candidate goes through Gemini with the scenario's specific prompt.
Severity is preserved so the downstream incident log can be color-coded.


In [ ]:
MEDIA_URL_MAP[VIDEO_NO] = "https://storage.googleapis.com/memories-test-data/test_1min.mp4"
media_url = resolve_media_url(VIDEO_NO)

SYSTEM = ("You are a security camera analyst. Be strict — only mark "
          "violation=true with clear visual evidence. Reply JSON only.")

incidents = []
for i, c in enumerate(candidates):
    s, h = c["scenario"], c["hit"]
    start, end = float(h["startTime"]), float(h["endTime"])
    raw = vlm_complete(
        f"Between {start:.0f}s and {end:.0f}s, {s.vlm_prompt}",
        video_url=media_url,
        system=SYSTEM,
    )
    try:
        verdict = json.loads(raw)
    except json.JSONDecodeError:
        verdict = {"raw": raw, "parse_error": True}
    verified = isinstance(verdict, dict) and verdict.get("violation") is True
    incidents.append({
        "scenario": s.name, "severity": s.severity,
        "video_no": h["videoNo"], "start": start, "end": end,
        "verdict": verdict, "verified": verified,
    })
    print(f"[{i+1}/{len(candidates)}] {s.name:<28}  {start:.0f}-{end:.0f}s  -> verified={verified}")


## Step 4 — severity rollup

The aggregation step. A real deployment would push critical incidents
through a webhook for real-time alerting and write the rest to a daily
report table.


In [ ]:
from collections import Counter

confirmed = [i for i in incidents if i["verified"]]
severity_counts = Counter(i["severity"] for i in confirmed)

print(f"\n{len(confirmed)} confirmed incidents (out of {len(incidents)} candidates):")
for sev in ("critical", "high", "medium", "low"):
    print(f"  {sev:<10} {severity_counts.get(sev, 0)}")

# Critical incidents would be pushed to your alerting webhook here.
for i in confirmed:
    if i["severity"] in ("critical", "high"):
        print(f"  ALERT: {i['scenario']} at {i['video_no']} {i['start']:.0f}-{i['end']:.0f}s")
